# Explore Baseline Model

This notebook demonstrates training a LightGBM baseline model with 3 features on synthetic fraud data.

In [ ]:
import sys
from pathlib import Path

# Add phase src to path
phase_dir = Path().cwd().parent
sys.path.insert(0, str(phase_dir / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from phase01_lightgbm_baseline.data_loader import DataLoader
from phase01_lightgbm_baseline.lightgbm_baseline import LightGBMWrapper

## Load Data

In [ ]:
loader = DataLoader()
train_df, test_df = loader.load_data(sample_limit=500)

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Fraud rate (train): {train_df['isFraud'].mean():.4f}")

## Prepare Features

In [ ]:
baseline_features = ["TransactionAmt", "time_since_creation", "n_prior_transactions"]

X_train = train_df[baseline_features].values
y_train = train_df["isFraud"].values
X_test = test_df[baseline_features].values
y_test = test_df["isFraud"].values

print(f"Features: {baseline_features}")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

## Train Model

In [ ]:
model = LightGBMWrapper(random_state=42)
metrics = model.full_pipeline(X_train, y_train, X_test, y_train, X_test, y_test, k=10)

print("Model metrics:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

## Visualize Predictions

In [ ]:
predictions = model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Prediction distribution
axes[0].hist(predictions, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Distribution')

# Predictions by label
sns.boxplot(x=y_test, y=predictions, ax=axes[1])
axes[1].set_xlabel('True Label')
axes[1].set_ylabel('Predicted Probability')
axes[1].set_title('Predictions by True Label')
axes[1].set_xticklabels(['Legit', 'Fraud'])

plt.tight_layout()
plt.show()

## Summary

We trained a LightGBM baseline model with 3 features. The model achieves reasonable performance on synthetic data. Next, we'll add graph features to see if they improve performance.